# 00 · Your first Claude call and workshop setup

**Level:** beginner · **Time:** 20–30 minutes · **Prerequisite:** basic Python.

We will use Claude for generation and a local embedding model for semantic retrieval.
The course uses a small **fictional Northstar Analytics** knowledge base, so you can
inspect every claim while demonstrating. No notebook requires another notebook's memory.

**You will learn to:** select the right kernel, load `CLAUDE_KEY` without displaying it,
inspect data, call Claude, and recognize the difference between a dry run and a live answer.

**Demo checkpoint:** run one cell at a time, explain its output, then change one input.


## 1. Install once, then choose the project kernel

From a terminal in the project folder, follow the README setup instructions. For the
tested Windows / Python 3.13 environment, the essential commands are:

```powershell
python -m venv .venv
.venv\Scripts\python.exe -m pip install -r requirements.lock.txt
.venv\Scripts\python.exe -m ipykernel install --sys-prefix --name rag-workshop --display-name "Python (RAG Workshop)"
.venv\Scripts\python.exe -m jupyter lab
```

For another Python/platform, use `requirements.txt` as the portable dependency list;
dependency resolution may choose different versions. In VS Code, open this notebook and
select **Python (RAG Workshop)** or the `.venv`
interpreter. Installing a package into a different Python does not install it into your kernel.
A first semantic-model run may download model weights; allow time before a live presentation.


In [ ]:
import sys
import importlib.metadata as metadata

print("Kernel Python:", sys.executable)
print("Python version:", sys.version.split()[0])
packages = ["anthropic", "python-dotenv", "pandas", "numpy", "scikit-learn",
            "sentence-transformers", "matplotlib", "pypdf", "ipykernel"]
for package in packages:
    try:
        print(f"{package:24} {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:24} MISSING — install requirements in this kernel")


## 2. Configure the key and execution mode

Your project already has `.env` with `CLAUDE_KEY`. The shared helper loads it; do not paste
a key into a cell or print environment variables. A new project would use `.env.example`
as its template. Keep the real `.env` out of version control.

Your current key has passed live checks through the Anthropic SDK and LangChain without
a workspace ID. If a different key returns a workspace-scoping error, add the ID from
your Anthropic Console workspace settings to `.env`:

```dotenv
ANTHROPIC_WORKSPACE_ID=wrkspc_your_workspace_id
```

`CLAUDE_WORKSPACE_ID` is also accepted; `ANTHROPIC_WORKSPACE_ID` takes precedence.
This setting is optional for keys already scoped to a workspace. The helpers pass it
as the `anthropic-workspace-id` request header; the ID is not a second API key.

`RAG_OFFLINE=1` skips Claude calls and model downloads. This is useful to rehearse the
Python/data flow, but an offline response is a **placeholder, never a model answer**.
Live mode uses your Anthropic credits. Change `.env`, then **restart the kernel** when
changing configuration. This avoids stale imported settings.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rag_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LangchainPractical project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag_workshop import (
    Document, load_documents, chunk_documents, TfidfIndex, DenseIndex,
    bm25_search, reciprocal_rank_fusion, context_text, ask_claude,
    rag_answer, show_hits, settings, offline_mode,
)
import pandas as pd
from IPython.display import display, Markdown

print("Mode:", "OFFLINE: Claude calls skipped" if offline_mode() else "LIVE: Claude calls use API credits")
print("Claude model:", settings().model)


## 3. Check configuration without revealing secrets

The next cell prints only whether a key exists and the chosen model names.
Model availability can differ by account. You can set `CLAUDE_MODEL` in `.env` if your
account needs a different model. We use Anthropic's SDK directly, so no OpenAI key is needed.


In [ ]:
config = settings()
print("Project:", config.root)
print("CLAUDE_KEY configured:", bool(config.api_key))
print("Anthropic workspace ID configured:", bool(config.workspace_id))
print("Claude model:", config.model)
print("Local embedding model:", config.embedding_model)
print("Knowledge base present:", (ROOT / "data" / "knowledge_base.json").exists())
if not offline_mode() and not config.api_key:
    raise RuntimeError("Add CLAUDE_KEY to the project .env, then restart this kernel.")
# Never display config itself: configuration objects may contain credentials.


## 4. Inspect the documents before adding an LLM

RAG means **retrieval augmented generation**: find relevant evidence, put it into the
prompt, and ask a generator to answer from it. Good retrieval starts with knowing the data.
Our `Document` has an `id`, `text`, and metadata such as title/source/department.


In [ ]:
documents = load_documents()
inventory = pd.DataFrame([
    {"id": d.id, "title": d.metadata.get("title"),
     "department": d.metadata.get("department"), "words": len(d.text.split())}
    for d in documents
])
display(inventory)
print("Documents:", len(documents))
print("Example source:", documents[0].metadata.get("source"))
print(documents[0].text)


## 5. Make a small Claude call

The system message sets the assistant's role; the user message requests an answer.
`max_tokens` limits output tokens, not input length. The helper returns response text,
token counts, latency, and cache/offline flags so you can explain the cost and behavior.
Repeated identical calls may be served from the workshop's local cache.

**Expected observation:** live mode gives two concise sentences; offline mode prints
a labeled skip message. Neither result contains your API key.


In [ ]:
response = ask_claude(
    "Explain retrieval augmented generation in exactly two short sentences for a Python beginner.",
    system="You are a patient instructor. Use plain language and no marketing claims.",
    max_tokens=160,
)
display(Markdown(response["text"]))
display(pd.DataFrame([{key: response.get(key) for key in
                       ["input_tokens", "output_tokens", "latency_s", "cached", "offline", "stop_reason"]}]))


## 6. Understand the direct SDK equivalent

This optional cell shows the underlying API instead of hiding it behind a helper.
It is off by default because it makes a second live request and bypasses the workshop cache.
Set `RUN_DIRECT_SDK_DEMO=True` to demonstrate it. We pass your existing `CLAUDE_KEY`
explicitly; the SDK's usual `ANTHROPIC_API_KEY` variable is not required here.


In [ ]:
RUN_DIRECT_SDK_DEMO = False
if RUN_DIRECT_SDK_DEMO and not offline_mode():
    import anthropic
    from rag_workshop import anthropic_headers
    client = anthropic.Anthropic(api_key=settings().api_key, default_headers=anthropic_headers())
    message = client.messages.create(
        model=settings().model,
        max_tokens=100,
        system="You are a concise teacher.",
        messages=[{"role": "user", "content": "What does retrieval mean in RAG?"}],
    )
    answer = "\n".join(block.text for block in message.content if block.type == "text")
    print(answer)
    print("Tokens:", message.usage.input_tokens, message.usage.output_tokens)
else:
    print("Direct SDK example skipped; enable it only when you want another live call.")


## 7. Optional: discover account-accessible Claude models

Model names evolve. The Models API lets you see the IDs available to your account.
This is a separate network operation and is opt-in. Copy a suitable ID to `CLAUDE_MODEL`
if needed, then restart the kernel. Do not print the client object or request headers.


In [ ]:
LIST_AVAILABLE_MODELS = False
if LIST_AVAILABLE_MODELS and not offline_mode():
    import anthropic
    from rag_workshop import anthropic_headers
    with anthropic.Anthropic(api_key=settings().api_key, default_headers=anthropic_headers()) as client:
        models = client.models.list(limit=100)
        display(pd.DataFrame([
            {"id": item.id, "display_name": item.display_name} for item in models.data
        ]))
else:
    print("Model listing skipped.")


## 8. Preview the entire RAG pipeline

This first preview uses lexical TF-IDF retrieval, which runs locally with no model download.
Later notebooks open up each step. Look at the retrieved source before reading Claude's
answer: is the answer really supported?


In [ ]:
chunks = chunk_documents(documents, chunk_size=90, overlap=20)
index = TfidfIndex(chunks)
question = "How many annual paid leave days do employees receive?"
hits = index.search(question, k=3)
display(show_hits(hits))
answer = rag_answer(question, hits, max_tokens=220)
display(Markdown(answer["text"]))


## 9. Demonstration exercise

**Try first:** change the question to ask about unused leave carry-forward. Predict the
relevant document, inspect the top three hits, then generate an answer. Identify the
number of days and deadline in the actual evidence.

**Instructor variation:** ask an unrelated question such as "What is the CEO's home
address?" The knowledge base does not contain it. A strong answer should say there is
insufficient evidence. Retrieval still returns nearest documents, even when none answer
the question; nearest does not mean relevant.


In [ ]:
# Sample solution. Change this question before running the cell again.
exercise_question = "How many leave days can I carry forward, and until when?"
exercise_hits = index.search(exercise_question, k=3)
display(show_hits(exercise_hits))
exercise_answer = rag_answer(exercise_question, exercise_hits, max_tokens=220)
display(Markdown(exercise_answer["text"]))
print("Check against source: up to 5 days, through March 31.")


## Troubleshooting and next step

| Symptom | Action |
|---|---|
| `ModuleNotFoundError` | Install `requirements.txt` into the selected kernel, then restart it. |
| Authentication failure | Check `CLAUDE_KEY` privately in `.env`; never paste the value into output. |
| Model not found or access denied | Use the optional Models API cell and set `CLAUDE_MODEL`. |
| HTTP 400 mentioning an unscoped key/workspace | Set `ANTHROPIC_WORKSPACE_ID` to your workspace ID in `.env`, then restart. |
| Rate limit / quota issue | Check account quota; wait before retrying. Do not loop unlimited calls. |
| Embedding download failure | Check connectivity; use `RAG_OFFLINE=1` for a lexical dry run. |
| Offline skip text appears | Remove/disable `RAG_OFFLINE` and restart for a live demonstration. |

**Ready to continue:** you can locate documents, explain the three RAG steps, and recognize
a real response versus a dry run. Open **01_rag_from_scratch.ipynb** next.

Official references: [Anthropic Python SDK](https://github.com/anthropics/anthropic-sdk-python),
[Messages API](https://platform.claude.com/docs/en/api/messages),
[Models API](https://platform.claude.com/docs/en/api/python/models),
[JupyterLab](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html).
